# 0. 패키지 설치 (평가 서버 환경과 동일하게)

In [ ]:
# Colab 환경용 패키지 설치
!pip install -q \
    transformers==4.57.3 \
    tokenizers==0.22.1 \
    accelerate==1.10.1 \
    datasets==4.4.1 \
    huggingface-hub==0.36.0 \
    safetensors==0.7.0 \
    sentencepiece==0.2.1 \
    compressed-tensors==0.13.0

print("[1/2] 기본 패키지 설치 완료")

In [ ]:
# llmcompressor 설치 (GPTQ 양자화용)
!pip install llmcompressor loguru pydantic -q

print("[2/2] llmcompressor 설치 완료")
print("\n⚠️ 런타임을 재시작한 후 다음 셀부터 실행하세요!")
print("   (런타임 → 런타임 다시 시작)")

# 1. 모델 설정
HuggingFace에서 EXAONE-4.0-1.2B 모델을 직접 다운로드합니다.

In [ ]:
# HuggingFace에서 모델 다운로드 설정
MODEL_ID = "LGAI-EXAONE/EXAONE-4.0-1.2B"
print(f"모델: {MODEL_ID}")
print("HuggingFace에서 모델을 다운로드합니다.")

# 2. Import

In [ ]:
import os
import torch
import shutil
from pathlib import Path

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("\n✅ Import 완료!")

# 3. Setting

In [ ]:
# 출력 디렉토리 설정
OUT_DIR = "./model"

# 데이터셋 설정
DATASET_ID = "LGAI-EXAONE/MANTA-1M"
DATASET_SPLIT = "train"

# 캘리브레이션 설정
NUM_CALIBRATION_SAMPLES = 256
MAX_SEQUENCE_LENGTH = 512

# 양자화 설정
SCHEME = "W4A16"  # 4-bit 가중치, 16-bit 활성화
TARGETS = ["Linear"]
IGNORE = ["embed_tokens", "lm_head"]

print(f"MODEL_ID: {MODEL_ID}")
print(f"OUT_DIR: {OUT_DIR}")
print(f"SCHEME: {SCHEME}")

# 4. Model Loads

In [ ]:
print("[INFO] 모델 로드 중...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

print(f"[INFO] 모델 파라미터: {model.num_parameters():,}")
print("[INFO] 모델/토크나이저 로드 완료")

# 5. Dataset Loads & Preprocess

In [ ]:
print("[INFO] 캘리브레이션 데이터 로드 중...")

ds = load_dataset(
    DATASET_ID,
    split=f"{DATASET_SPLIT}[:{NUM_CALIBRATION_SAMPLES}]",
)

def preprocess(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["conversations"],
            add_generation_prompt=True,
            tokenize=False)
    }

ds = ds.map(preprocess)

print(f"[INFO] 데이터셋 크기: {len(ds)}")
print("[INFO] 데이터 전처리 완료")

# 6. GPTQ Quantization

In [ ]:
print(f"[INFO] GPTQ 양자화 시작")
print(f"  - scheme: {SCHEME}")
print(f"  - samples: {NUM_CALIBRATION_SAMPLES}")
print(f"  - max_len: {MAX_SEQUENCE_LENGTH}")
print(f"  - ignore: {IGNORE}")
print("\n이 과정은 10-20분 정도 소요됩니다...\n")

recipe = [
    GPTQModifier(
        scheme=SCHEME,
        targets=TARGETS,
        ignore=IGNORE,
    )
]

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
)

print("\n[INFO] GPTQ 양자화 완료!")

# 7. Model Save

In [ ]:
print("[INFO] 모델 저장 중...")

os.makedirs(OUT_DIR, exist_ok=True)

model.save_pretrained(OUT_DIR, save_compressed=True)
tokenizer.save_pretrained(OUT_DIR)

# 저장된 파일 확인
print(f"\n[INFO] 저장된 파일:")
for f in os.listdir(OUT_DIR):
    size = os.path.getsize(os.path.join(OUT_DIR, f))
    print(f"  {f}: {size/1e6:.1f} MB")

print(f"\n[INFO] 모델 저장 완료: {OUT_DIR}")

# 8. Submission 파일 생성

In [ ]:
zip_name = "submit"
print(f"[INFO] {zip_name}.zip 생성 중...")

shutil.make_archive(
    base_name=zip_name,
    format="zip",
    root_dir=".",
    base_dir=OUT_DIR,
)

zip_size = os.path.getsize(f"{zip_name}.zip") / 1e9
print(f"[INFO] 생성 완료: {zip_name}.zip ({zip_size:.2f} GB)")

# 9. 다운로드 (Colab)

In [ ]:
from google.colab import files

print("submit.zip 다운로드 시작...")
files.download("submit.zip")
print("\n✅ 다운로드 완료! DACON에 제출하세요.")

# 10. (선택) 모델 테스트

In [ ]:
# 양자화된 모델로 간단한 테스트
print("[INFO] 양자화된 모델 테스트...")

message = [{"role": "user", "content": "안녕하세요, 자기소개 해주세요."}]
input_ids = tokenizer.apply_chat_template(
    message, 
    add_generation_prompt=True, 
    return_tensors="pt"
).to(model.device)

output = model.generate(input_ids, max_new_tokens=100, do_sample=False)
response = tokenizer.decode(output[0], skip_special_tokens=True)

print(f"\n응답:\n{response}")